<a href="https://colab.research.google.com/github/ghanha0a/111/blob/main/bone_fracture_model_firstCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [2]:
from google.colab import files

# Upload the kaggle.json file
files.upload()

# Create a .kaggle directory
!mkdir -p ~/.kaggle

# Move the kaggle.json file to the .kaggle directory
# Make sure you have uploaded kaggle.json to your Colab environment's session storage
!mv kaggle.json ~/.kaggle/

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [3]:
!kaggle datasets download bmadushanirodrigo/fracture-multi-region-x-ray-data

Dataset URL: https://www.kaggle.com/datasets/bmadushanirodrigo/fracture-multi-region-x-ray-data
License(s): ODC Public Domain Dedication and Licence (PDDL)
100% 481M/481M [00:07<00:00, 66.6MB/s]



In [4]:
!unzip fracture-multi-region-x-ray-data.zip -d '/content/drive/MyDrive/ColabNotebooks'

Streaming output truncated to the last 5000 lines.
  inflating: /content/drive/MyDrive/ColabNotebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3 (1).jpg  
  inflating: /content/drive/MyDrive/ColabNotebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3-rotated1 (1).jpg  
  inflating: /content/drive/MyDrive/ColabNotebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3-rotated1.jpg  
  inflating: /content/drive/MyDrive/ColabNotebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2-rotated3.jpg  
  inflating: /content/drive/MyDrive/ColabNotebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/14-rotated2-rotated2.jpg  
  inflating: /content/drive/MyDrive/C

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

img_width, img_height = 150, 150

# Create ImageDataGenerator for data augmentation and preprocessing
train_datagen = ImageDataGenerator()
validation_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory('/content/drive/MyDrive/Colab Notebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train',
                                                      target_size=(img_width, img_height),
                                                      batch_size=32,
                                                      class_mode='binary') # 'binary' for binary classification


# Flow training images in batches of 32 using train_datagen

# Flow validation images in batches of 32 using validation_datagen
validation_generator = validation_datagen.flow_from_directory('/content/drive/MyDrive/Colab Notebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/val',
                                                              target_size=(img_width, img_height),
                                                              batch_size=32,
                                                              class_mode='binary')

# Flow test images in batches of 32 using test_datagen (if test_dir is provided)
test_generator = test_datagen.flow_from_directory('/content/drive/MyDrive/Colab Notebooks/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/test', # If you have a separate test directory
                                                  target_size=(img_width, img_height),
                                                  batch_size=32,
                                                  class_mode='binary', # For binary classification
                                                  shuffle=False) # Keep data in order for evaluation

model = keras.Sequential()
model.add(keras.layers.Conv2D(32,(3,3), activation='relu', input_shape=(img_width,img_height,3)))
model.add(keras.layers.MaxPooling2D((2,2)))
model.add(keras.layers.Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.MaxPooling2D((2,2)))
model.add(keras.layers.Conv2D(64,(3,3), activation='relu'))
model.add(keras.layers.MaxPooling2D((2,2)))

model.summary()
model.add(keras.layers.Flatten())

model.add(keras.layers.Dense(256, activation='relu'))
model.add(keras.layers.Dense(128, activation='relu'))
model.add(keras.layers.Dense(1, activation='sigmoid'))
model.summary()

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_generator, epochs=10)

loss, accuracy = model.evaluate(test_generator)

Found 9246 images belonging to 2 classes.
Found 829 images belonging to 2 classes.
Found 506 images belonging to 2 classes.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 34, 34, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,320 (220.00 KB)

 Trainable params: 56,320 (220.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 34, 34, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │     4,735,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,824,577 (18.40 MB)

 Trainable params: 4,824,577 (18.40 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 53/289 ━━━━━━━━━━━━━━━━━━━━ 43s 184ms/step - accuracy: 0.6316 - loss: 19.3819

In [6]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Upload X-Ray Image</title>
</head>
<body>
    <h1>Upload X-Ray Image for Fracture Detection</h1>
    <form id="upload-form" enctype="multipart/form-data">
        <input type="file" name="xray_image" accept="image/*">
        <input type="submit" value="Predict">
    </form>

    <div id="result">
        <!-- Prediction results will be displayed here -->
    </div>

    <script>
        document.getElementById('upload-form').addEventListener('submit', async function(event) {
            event.preventDefault();

            const formData = new FormData(this);
            // In a real application, you would send this to a server endpoint.
            // For demonstration, we'll just show what the form data contains.
            const resultDiv = document.getElementById('result');
            resultDiv.innerHTML = '<h2>Uploading image...</h2>';

            // Example of what would happen if you had a server endpoint:
            // try {
            //     const response = await fetch('/predict', {
            //         method: 'POST',
            //         body: formData
            //     });
            //     const data = await response.json();
            //     resultDiv.innerHTML = `<h2>Prediction: ${data.prediction}</h2>`;
            // } catch (error) {
            //     resultDiv.innerHTML = `<h2>Error: ${error}</h2>`;
            // }

            resultDiv.innerHTML = `<h2>Image selected for upload. (To get a prediction, run the Python code below in Colab after uploading.)</h2>`;
        });
    </script>
</body>
</html>
"""

print(html_content)
# You can copy this output and save it as an 'upload.html' file on your computer.


<!DOCTYPE html>
<html>
<head>
    <title>Upload X-Ray Image</title>
</head>
<body>
    <h1>Upload X-Ray Image for Fracture Detection</h1>
    <form id="upload-form" enctype="multipart/form-data">
        <input type="file" name="xray_image" accept="image/*">
        <input type="submit" value="Predict">
    </form>

    <div id="result">
        <!-- Prediction results will be displayed here -->
    </div>

    <script>
        document.getElementById('upload-form').addEventListener('submit', async function(event) {
            event.preventDefault();

            const formData = new FormData(this);
            // In a real application, you would send this to a server endpoint.
            // For demonstration, we'll just show what the form data contains.
            const resultDiv = document.getElementById('result');
            resultDiv.innerHTML = '<h2>Uploading image...</h2>';

            // Example of what would happen if you had a server endpoint:
            // try {
      

In [16]:
from google.colab import files
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Save the model first to ensure it exists if this cell is run out of order
# This assumes 'model' is available from a previous cell where it was defined and trained.
model.save('/content/bone_fracture_model.h5')

# Load the saved model
loaded_model = load_model('/content/drive/MyDrive/Colab Notebooks/MY PROJECTS/ML&DL/deep learning/اهم مودل/bone_fracture_model_firstCNN.h5')

# Function to preprocess the image
def preprocess_image(img_path, target_size=(150, 150)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Create a batch dimension
    img_array = img_array / 255.0  # Normalize pixel values to [0, 1]
    return img_array

# Function to make a prediction
def predict_image(img_path):
    processed_img = preprocess_image(img_path)
    prediction = loaded_model.predict(processed_img)
    # For binary classification with sigmoid activation, output is a probability
    # If > 0.5, classify as fractured, otherwise not fractured
    if prediction[0][0] > 0.5:
        return "Fractured", prediction[0][0]
    else:
        return "Not Fractured", prediction[0][0]

# Upload an image
print("Please upload an image to predict:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'User uploaded file "{fn}"')
    predicted_class, probability = predict_image(fn)
    print(f"Prediction for {fn}: {predicted_class} (Probability: {probability:.4f})")

Please upload an image to predict:


Saving 87-rotated3-rotated3-rotated2.jpg to 87-rotated3-rotated3-rotated2.jpg
User uploaded file "87-rotated3-rotated3-rotated2.jpg"
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 449ms/step
Prediction for 87-rotated3-rotated3-rotated2.jpg: Not Fractured (Probability: 0.3123)
